# Lab22 — Publish reusable adapters to HF + transfer to Kaggle

CPU runtime is enough. Add a new write token to Colab Secrets as HF_TOKEN and enable notebook access. The notebook creates a **public Model** repo with SFT/DPO adapters and a model card, plus a **private Dataset** for Kaggle transfer. No merged model upload is needed. Use the Drive folder containing actual adapter weights, not the submission zip.

In [ ]:
DRIVE_FOLDER = "Lab22_PhamHoangTrong_2A202602765"
MODEL_NAME = "lab22-dpo-experimental"

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, sys, subprocess
drive.mount("/content/drive")
WORK = Path("/content/drive/MyDrive") / DRIVE_FOLDER
assert (WORK / "adapters/sft-mini/adapter_model.safetensors").exists(), "Không thấy adapter SFT"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

In [ ]:
local_scripts = Path("/content/lab22-hf-tools")
local_scripts.mkdir(exist_ok=True)
(local_scripts / 'publish_hf.py').write_text('"""Publish reusable SFT+DPO adapters/model card and a private Kaggle transfer repo.\n\nCredentials come only from HF_TOKEN environment variable. Run beside Drive data.\n"""\nimport argparse\nimport json\nimport os\nimport shutil\nfrom pathlib import Path\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--root\', required=True)\n    parser.add_argument(\'--name\', default=\'lab22-dpo-experimental\')\n    args = parser.parse_args()\n    root = Path(args.root).resolve()\n    from huggingface_hub import HfApi\n    token = os.environ.get(\'HF_TOKEN\')\n    if not token:\n        raise RuntimeError(\'Set HF_TOKEN using Colab Secrets, not notebook source.\')\n    for name in (\'sft-mini\', \'dpo\'):\n        for file in (\'adapter_model.safetensors\', \'adapter_config.json\'):\n            p = root / \'adapters\' / name / file\n            assert p.exists() and p.stat().st_size, f\'Missing/empty {p}\'\n    api = HfApi(token=token)\n    username = api.whoami()[\'name\']\n    model_id = f\'{username}/{args.name}\'\n    transfer_id = f\'{username}/lab22-continue\'\n    staging = Path(\'/tmp/lab22-hf-publish\')\n    staging.mkdir(parents=True, exist_ok=True)\n    # Only copy needed adapter/tokenizer files. No datasets, cached references or keys.\n    for name, destination in ((\'dpo\', staging), (\'sft-mini\', staging / \'sft_adapter\')):\n        destination.mkdir(exist_ok=True)\n        for p in (root / \'adapters\' / name).iterdir():\n            if p.is_file() and (p.suffix in (\'.json\', \'.safetensors\') or p.name == \'tokenizer.model\'):\n                shutil.copy2(p, destination / p.name)\n    cfg = json.loads((staging / \'adapter_config.json\').read_text(encoding=\'utf-8\'))\n    cfg[\'base_model_name_or_path\'] = \'./sft-merged\'\n    (staging / \'adapter_config.json\').write_text(json.dumps(cfg, indent=2), encoding=\'utf-8\')\n    source_dir = Path(__file__).resolve().parent\n    shutil.copy2(source_dir / \'reuse_hf_adapter.py\', staging / \'reuse_hf_adapter.py\')\n    shutil.copy2(root / \'requirements.txt\', staging / \'requirements.txt\')\n    metrics_dir = staging / \'evaluation\'\n    metrics_dir.mkdir(exist_ok=True)\n    for relative in (\'adapters/dpo/dpo_metrics.json\', \'data/eval/judge_summary.json\', \'data/eval/sft_metrics.json\'):\n        source = root / relative\n        if source.exists():\n            shutil.copy2(source, metrics_dir / source.name)\n    card = (root / \'submission/MODEL-CARD.md\').read_text(encoding=\'utf-8\')\n    card += f\'\'\'\n\n## Reuse from Hugging Face\n\nModel repo: https://huggingface.co/{model_id}\n\nThe root LoRA is DPO. `sft_adapter/` contains the preceding SFT adapter.\nThe DPO reference is **merged SFT**, not the original base model. Therefore\nloading the root adapter directly on the base model is incorrect. The bundled\nhelper reconstructs SFT locally, updates a local copy of the DPO config, and\nloads the final model. CUDA GPU and sufficient temporary disk are required.\n\n```python\nfrom huggingface_hub import hf_hub_download\nimport subprocess, sys\nhelper = hf_hub_download("{model_id}", "reuse_hf_adapter.py")\nsubprocess.run([sys.executable, helper, "--repo", "{model_id}",\n                "--prompt", "Giải thích quicksort trong 5 câu."], check=True)\n```\n\nInstall the training packages in `requirements.txt` before running the helper.\nThe helper does no new training; it merges the saved SFT LoRA before applying DPO.\nReference reconstruction/reuse is provided as code, not a tested deployment guarantee.\nKnown output defects: stray tool_call markers and imperfect instruction following.\nThe model card and evaluation files report the actual lab run, not benchmark claims.\nDo not infer a redistribution license for preference data or derivative weights.\n\'\'\'\n    (staging / \'README.md\').write_text(card, encoding=\'utf-8\')\n    api.create_repo(repo_id=model_id, repo_type=\'model\', private=False, exist_ok=True)\n    api.upload_folder(repo_id=model_id, repo_type=\'model\', folder_path=str(staging),\n                      commit_message=\'Publish saved SFT/DPO adapters, model card and reuse helper\')\n    published = set(api.list_repo_files(model_id, repo_type=\'model\'))\n    for file in (\'adapter_model.safetensors\', \'adapter_config.json\', \'sft_adapter/adapter_model.safetensors\', \'README.md\', \'reuse_hf_adapter.py\'):\n        assert file in published, f\'Upload verification failed: {file}\'\n    print(\'Model uploaded:\', f\'https://huggingface.co/{model_id}\', flush=True)\n    # Separate private transfer dataset: preserve all adapters and evaluation evidence.\n    api.create_repo(repo_id=transfer_id, repo_type=\'dataset\', private=True, exist_ok=True)\n    info = api.repo_info(transfer_id, repo_type=\'dataset\')\n    if not info.private:\n        raise RuntimeError(\'Transfer dataset is public; set it private before uploading data.\')\n    for folder in (\'adapters\', \'data\', \'notebooks\', \'submission\'):\n        path = root / folder\n        if path.exists():\n            api.upload_folder(repo_id=transfer_id, repo_type=\'dataset\', folder_path=str(path), path_in_repo=folder,\n                allow_patterns=[\'*.safetensors\', \'*.json\', \'*.jsonl\', \'*.parquet\', \'*.ipynb\', \'*.md\', \'*.txt\', \'*.png\', \'*.jpg\'],\n                ignore_patterns=[\'**/.ipynb_checkpoints/**\', \'**/ref/**\', \'**/*-checkpoints/**\'],\n                commit_message=f\'Transfer Lab22 {folder} for Kaggle continuation\')\n    expected = {\'adapters/sft-mini/adapter_model.safetensors\', \'adapters/dpo/adapter_model.safetensors\', \'data/pref/eval.parquet\'}\n    assert expected <= set(api.list_repo_files(transfer_id, repo_type=\'dataset\')), \'Transfer incomplete.\'\n    evidence = {\'adapter\': model_id, \'reference\': \'reconstruct from sft_adapter/ using reuse_hf_adapter.py\',\n                \'model_url\': f\'https://huggingface.co/{model_id}\', \'transfer_dataset\': transfer_id,\n                \'upload_verified\': True, \'reuse_gpu_tested\': False}\n    (root / \'data/eval/hub_upload.json\').write_text(json.dumps(evidence, indent=2), encoding=\'utf-8\')\n    reflection = root / \'submission/REFLECTION.md\'\n    text = reflection.read_text(encoding=\'utf-8\')\n    text = text.replace(\'- [ ] HF Hub (+3)\', \'- [x] HF Hub (+3)\')\n    note = f\'\\n\\n## HF Hub — adapter và model card\\n\\nĐã upload và kiểm tra các file trên [{model_id}](https://huggingface.co/{model_id}). Repo chứa adapter DPO, adapter SFT, model card (mô hình gốc, dữ liệu, siêu tham số, kết quả thật) và script khôi phục SFT để tái sử dụng. Repo dataset {transfer_id} là gói riêng tư phục vụ chuyển sang Kaggle, không thay thế model card. Chưa kiểm thử suy luận GPU từ repo HF sau upload.\\n\'\n    if \'## HF Hub — adapter và model card\' not in text:\n        text += note\n    reflection.write_text(text, encoding=\'utf-8\')\n    print(\'Kaggle HF_DATASET_ID =\', transfer_id, flush=True)\n    print(\'Upload complete; download hub_upload.json and updated REFLECTION.md for GitHub submission.\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")
(local_scripts / 'reuse_hf_adapter.py').write_text('"""Reconstruct SFT from its saved adapter, then load DPO. Requires CUDA.\n\npython reuse_hf_adapter.py --repo account/lab22-dpo-experimental --prompt "Xin chào"\n"""\nimport argparse\nimport json\nimport shutil\nfrom pathlib import Path\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--repo\', required=True)\n    parser.add_argument(\'--work\', default=\'./lab22-hf-runtime\')\n    parser.add_argument(\'--prompt\', default=\'Giải thích thuật toán quicksort trong 5 câu.\')\n    args = parser.parse_args()\n    import unsloth\n    import torch\n    from huggingface_hub import snapshot_download\n    from unsloth import FastLanguageModel\n    assert torch.cuda.is_available(), \'A CUDA GPU is required.\'\n    work = Path(args.work).resolve()\n    snapshot = Path(snapshot_download(args.repo))\n    sft = work / \'sft-merged\'\n    dpo = work / \'dpo-adapter\'\n    work.mkdir(parents=True, exist_ok=True)\n    if not (sft / \'config.json\').exists():\n        model, tokenizer = FastLanguageModel.from_pretrained(\n            model_name=str(snapshot / \'sft_adapter\'), max_seq_length=768, load_in_4bit=True,\n        )\n        model.save_pretrained_merged(str(sft), tokenizer, save_method=\'merged_16bit\')\n        del model, tokenizer\n        import gc\n        gc.collect()\n        torch.cuda.empty_cache()\n    dpo.mkdir(exist_ok=True)\n    for p in snapshot.iterdir():\n        if p.is_file() and (p.suffix in (\'.json\', \'.safetensors\') or p.name == \'tokenizer.model\'):\n            shutil.copy2(p, dpo / p.name)\n    cfg = json.loads((dpo / \'adapter_config.json\').read_text(encoding=\'utf-8\'))\n    cfg[\'base_model_name_or_path\'] = str(sft)\n    (dpo / \'adapter_config.json\').write_text(json.dumps(cfg, indent=2), encoding=\'utf-8\')\n    model, tokenizer = FastLanguageModel.from_pretrained(model_name=str(dpo), max_seq_length=768, load_in_4bit=True)\n    FastLanguageModel.for_inference(model)\n    prompt = tokenizer.apply_chat_template([{\'role\': \'user\', \'content\': args.prompt}],\n        tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    inputs = tokenizer(prompt, return_tensors=\'pt\', add_special_tokens=False).to(model.device)\n    with torch.no_grad():\n        output = model.generate(**inputs, max_new_tokens=384, do_sample=False, pad_token_id=tokenizer.pad_token_id)\n    print(tokenizer.decode(output[0, inputs[\'input_ids\'].shape[1]:], skip_special_tokens=True))\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding="utf-8")
subprocess.run([sys.executable, "-u", str(local_scripts / "publish_hf.py"), "--root", str(WORK), "--name", MODEL_NAME], check=True)

In [ ]:
from google.colab import files
import json
evidence = json.loads((WORK / "data/eval/hub_upload.json").read_text())
print("Model:", evidence["model_url"])
print("Kaggle HF_DATASET_ID:", evidence["transfer_dataset"])
files.download(str(WORK / "data/eval/hub_upload.json"))
files.download(str(WORK / "submission/REFLECTION.md"))